In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     RandomizedSearchCV)
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.preprocessing import label_binarize

In [2]:
df = pd.read_csv('../data/processed/df_cleaned.csv')

with open('../src/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

with open('../src/features.pkl', 'rb') as f:
    FEATURES = pickle.load(f)

X = df[FEATURES]
y = df['risk_encoded']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train_scaled = scaler.transform(X_train)
X_test_scaled  = scaler.transform(X_test)

CLASS_NAMES = ['Low', 'Medium', 'High']

print(f"X_train : {X_train_scaled.shape}")
print(f"X_test : {X_test_scaled.shape}")

X_train : (36000, 19)
X_test : (9000, 19)


In [4]:
param_dist = {
    'n_estimators' : [50, 100, 200, 300],
    'max_depth' : [5, 10, 15, 20, None],
    'min_samples_leaf' : [1, 2, 5, 10],
    'min_samples_split' : [2, 5, 10],
    'max_features' : ['sqrt', 'log2', None]
}

print("Parameter grid:")
for param, values in param_dist.items():
    print(f"{param:<20}: {values}")

total = 1
for v in param_dist.values():
    total *= len(v)
print(f"\nTotal combinations : {total}")
print(f"RandomizedSearchCV : will try 20 combinations only")

Parameter grid:
n_estimators        : [50, 100, 200, 300]
max_depth           : [5, 10, 15, 20, None]
min_samples_leaf    : [1, 2, 5, 10]
min_samples_split   : [2, 5, 10]
max_features        : ['sqrt', 'log2', None]

Total combinations : 720
RandomizedSearchCV : will try 20 combinations only


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_base = RandomForestClassifier(random_state=42, class_weight='balanced')

random_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_dist,
    n_iter=20,
    scoring='accuracy',
    cv=cv,
    random_state=42,
    n_jobs=-1,       
    verbose=1
)

random_search.fit(X_train_scaled, y_train)

print(f"\nBest parameters:")
for param, value in random_search.best_params_.items():
    print(f"{param:<20}: {value}")
print(f"\nBest CV Accuracy : {random_search.best_score_:.4f}")

Fitting 5 folds for each of 20 candidates, totalling 100 fits

Best parameters:
n_estimators        : 200
min_samples_split   : 5
min_samples_leaf    : 1
max_features        : None
max_depth           : 15

Best CV Accuracy : 0.9999


In [6]:
best_rf = random_search.best_estimator_

y_pred = best_rf.predict(X_test_scaled)
y_prob = best_rf.predict_proba(X_test_scaled)

acc = accuracy_score(y_test, y_pred)
roc = roc_auc_score(
    label_binarize(y_test, classes=[0,1,2]),
    y_prob, multi_class='ovr', average='weighted'
)

print(f"Tuned Random Forest — Test Set Results")
print(f"{'-'*45}")
print(f"Accuracy : {acc:.4f}")
print(f"ROC-AUC  : {roc:.4f}")
print(f"\n{classification_report(y_test, y_pred, target_names=CLASS_NAMES)}")

# Bandingkan dengan model sebelumnya
with open('../src/model.pkl', 'rb') as f:
    base_model = pickle.load(f)

base_pred = base_model.predict(X_test_scaled)
base_acc  = accuracy_score(y_test, base_pred)

print(f"{'-'*45}")
print(f"Comparison:")
print(f"Base RF Accuracy: {base_acc:.4f}")
print(f"Tuned RF Accuracy: {acc:.4f}")
print(f"Improvement: {(acc - base_acc):.4f}")

Tuned Random Forest — Test Set Results
---------------------------------------------
Accuracy : 1.0000
ROC-AUC  : 1.0000

              precision    recall  f1-score   support

         Low       1.00      1.00      1.00      3000
      Medium       1.00      1.00      1.00      3000
        High       1.00      1.00      1.00      3000

    accuracy                           1.00      9000
   macro avg       1.00      1.00      1.00      9000
weighted avg       1.00      1.00      1.00      9000

---------------------------------------------
Comparison:
Base RF Accuracy: 0.9999
Tuned RF Accuracy: 1.0000
Improvement: 0.0001


In [7]:
with open('../src/model.pkl', 'wb') as f:
    pickle.dump(best_rf, f)

# Save best params untuk dokumentasi
import json
with open('../src/best_params.json', 'w') as f:
    json.dump(random_search.best_params_, f, indent=2)
    
print(f"\nFinal model parameters:")
for param, value in random_search.best_params_.items():
    print(f"  {param:<20}: {value}")


Final model parameters:
  n_estimators        : 200
  min_samples_split   : 5
  min_samples_leaf    : 1
  max_features        : None
  max_depth           : 15
